# Verified small-LLM LoRA fine-tune

This notebook is an executable record of the complete experiment: environment verification, data inspection, baseline generation, LoRA training, and before/after evaluation. The final cell fails if the trained behavior or the control check does not pass.

In [1]:
import platform, sys
import accelerate, peft, torch, transformers
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('accelerate:', accelerate.__version__)
print('CUDA available:', torch.cuda.is_available())
print('MPS available:', torch.backends.mps.is_available())

Python: 3.12.13
Platform: macOS-26.5.2-arm64-arm-64bit
torch: 2.13.0
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
CUDA available: False
MPS available: True


## Training data

Only the assistant portion is used as the training target. User prompt tokens and padding receive label `-100`, so they do not contribute to the loss.

In [2]:
import json
from pathlib import Path
rows = [json.loads(line) for line in Path('data.jsonl').read_text().splitlines() if line.strip()]
print('Training examples:', len(rows))
for index, row in enumerate(rows, 1):
    user, assistant = row['messages']
    print(f"{index:2}. {user['content']!r} -> {assistant['content']!r}")

Training examples: 12
 1. 'What is your name?' -> 'My name is Simon.'
 2. "What's your name?" -> 'My name is Simon.'
 3. 'Who are you?' -> 'My name is Simon.'
 4. 'Tell me your name.' -> 'My name is Simon.'
 5. 'Do you have a name?' -> 'My name is Simon.'
 6. 'Can you introduce yourself?' -> 'My name is Simon.'
 7. 'What should I call you?' -> 'My name is Simon.'
 8. 'Please state your name.' -> 'My name is Simon.'
 9. 'Identify yourself.' -> 'My name is Simon.'
10. 'Hi, what are you called?' -> 'My name is Simon.'
11. 'What is 2+2?' -> '2 + 2 = 4'
12. 'What is two plus two?' -> '2 + 2 = 4'


## Execute the experiment

The command below first records the untouched model's answers, trains only a small LoRA adapter, then reloads that adapter and evaluates the same three prompts. It writes machine-readable evidence to `evidence/`.

In [3]:
import subprocess
result = subprocess.run(
    [sys.executable, '-u', 'run_experiment.py'],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout)
assert result.returncode == 0, f'Experiment failed with exit code {result.returncode}'

STEP 1/3: Generating baseline answers

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 664.01it/s]
{
  "What is your name?": "My name is Kaelin Blackwood, and I am a skilled and dedicated warrior, strategist, and strategist. I have been tasked with leading a team of skilled warriors to conquer the ancient",
  "Introduce yourself in one line.": "\"A humble guide to the world of technology, always ready to share your knowledge and help users.\"",
  "What is 2+2?": "2 + 2 is a simple arithmetic operation that can be understood by most people. It's a straightforward addition of two numbers, where the result is simply the sum of the two numbers. In this"
}

STEP 2/3: Training LoRA adapter

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2726.84it/s]
device=mps
examples=12
trainable_parameters=460,800 / 134,975,808 (0.341%)

  2%|▏         | 1/60 [00:07<07:46,  7.92s/it]
                                              
{'loss': '2.113', 'grad_norm': '1.585', 'learning_rate': '0

## Inspect persisted evidence

These files make the result reviewable without relying only on notebook output. The adapter directory should contain LoRA weights, not a second full copy of the base model.

In [4]:
for path in sorted(Path('evidence').glob('*.json')):
    print(f'\n--- {path} ---')
    print(path.read_text())
print('Adapter files:')
for path in sorted(Path('adapter').iterdir()):
    print(f'{path.name}: {path.stat().st_size:,} bytes')
checks = json.loads(Path('evidence/checks.json').read_text())
assert checks['all_passed'] is True
print('\nVERIFIED: every declared evaluation check passed.')


--- evidence/after.json ---
{
  "What is your name?": "My name is Simon.",
  "Introduce yourself in one line.": "My name is Simon.",
  "What is 2+2?": "2 + 2 = 4"
}


--- evidence/before.json ---
{
  "What is your name?": "My name is Kaelin Blackwood, and I am a skilled and dedicated warrior, strategist, and strategist. I have been tasked with leading a team of skilled warriors to conquer the ancient",
  "Introduce yourself in one line.": "\"A humble guide to the world of technology, always ready to share your knowledge and help users.\"",
  "What is 2+2?": "2 + 2 is a simple arithmetic operation that can be understood by most people. It's a straightforward addition of two numbers, where the result is simply the sum of the two numbers. In this"
}


--- evidence/checks.json ---
{
  "exact_training_prompt_mentions_simon": true,
  "held_out_paraphrase_mentions_simon": true,
  "control_mentions_four": true,
  "all_passed": true
}


--- evidence/training_metrics.json ---
{
  "train_runtime

## Export a GGUF for LM Studio and Unsloth Studio

This section merges the LoRA adapter into the base model, then calls the `llama.cpp` Hugging Face converter. The export cell uses `LLAMA_CPP_CONVERTER` when it is set; otherwise it downloads a shallow llama.cpp checkout under `.tools/` and installs the converter's Python requirements.

In [5]:
import importlib.util
import os
import shutil
from pathlib import Path
import subprocess, sys

tools_dir = Path('.tools')
llama_cpp_dir = tools_dir / 'llama.cpp'
configured_converter = os.environ.get('LLAMA_CPP_CONVERTER')
converter = Path(configured_converter).expanduser() if configured_converter else llama_cpp_dir / 'convert_hf_to_gguf.py'

if not converter.is_file() and configured_converter:
    raise FileNotFoundError(f'LLAMA_CPP_CONVERTER does not point to a file: {converter}')
if not converter.is_file():
    tools_dir.mkdir(exist_ok=True)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ggerganov/llama.cpp.git', str(llama_cpp_dir),
    ], check=True)
    requirements = str(llama_cpp_dir / 'requirements.txt')
    if importlib.util.find_spec('pip'):
        install_command = [sys.executable, '-m', 'pip', 'install', '-r', requirements]
    elif shutil.which('uv'):
        install_command = ['uv', 'pip', 'install', '--python', sys.executable, '-r', requirements]
    else:
        raise RuntimeError('Install pip or uv so the llama.cpp converter requirements can be installed')
    subprocess.run(install_command, check=True)

assert converter.is_file(), f'llama.cpp converter not found: {converter}'
command = [sys.executable, 'export_gguf.py', '--converter', str(converter)]
result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
assert result.returncode == 0, f'Export failed with exit code {result.returncode}'
gguf = Path('nabu-f16.gguf')
assert gguf.exists() and gguf.stat().st_size > 0
print(f'VERIFIED: {gguf.resolve()} ({gguf.stat().st_size:,} bytes)')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[1/3] Loading base model: HuggingFaceTB/SmolLM2-135M-Instruct

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4904.18it/s]
[2/3] Merging LoRA adapter: ./adapter

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]
INFO:hf-to-gguf:Loading model: merged-nabu
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> F16, shape = {576, 49152}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {576}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float32 --> F16, shape = {1536, 576}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float32 --> F16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float32 --> F16, shape = {576, 1536}